In [2]:
"""
Lake ice freeze-up / break-up data from SYKE (Finnish Environment Institute)

Output (one row per observation site x winter season):
    icebreak_data.csv         analysis dataset with outcomes + covariates
    icebreak_data_places.csv  one row per site (record length, lake info)

Data: SYKE Hydrologiarajapinta 1.2 + Jarvirajapinta 1.0, CC BY 4.0
    https://rajapinnat.ymparisto.fi/api/Hydrologiarajapinta/1.2/
    https://rajapinnat.ymparisto.fi/api/jarvirajapinta/1.0/

Requirements: pip install requests pandas numpy
"""

import numpy as np
import pandas as pd
import requests

# ---- Settings ---------------------------------------------------------------

MIN_SEASONS = 30    # keep sites with at least this many break-up years
MIN_SEASON = 1900   # drop very old records (set to None to keep everything)

HYDRO_API = "https://rajapinnat.ymparisto.fi/api/Hydrologiarajapinta/1.2/odata"
LAKE_API = "https://rajapinnat.ymparisto.fi/api/jarvirajapinta/1.0/odata"

# Lippu_id codes in the JaatJaanlahto table (from the Lippu table):
#   27 = b5  ice gone from the open waters further out (partial break-up)
#   28 = b4  ice gone from the whole visible area      (FULL BREAK-UP)
#   29 = x5  whole visible area frozen                  (first freeze-up)
#   30 = x7  whole visible area permanently frozen      (PERMANENT FREEZE-UP)
EVENT_CODES = {
    27: "breakup_partial",
    28: "breakup",
    29: "freeze_first",
    30: "freeze",
}


# ---- Helpers ----------------------------------------------------------------

def get_all_odata(url, params=None):
    """Download every page of an OData collection into a DataFrame."""
    rows = []
    session = requests.Session()
    session.headers["Accept"] = "application/json"
    while url:
        r = session.get(url, params=params, timeout=120)
        r.raise_for_status()
        data = r.json()
        rows.extend(data.get("value", []))
        url = data.get("odata.nextLink")
        params = None  # the next link already contains the query
    return pd.DataFrame(rows)


def dms_to_decimal(x):
    """Paikka coordinates are strings like '613601' = 61 deg 36' 01"."""
    x = pd.to_numeric(x, errors="coerce")
    deg = x // 10000
    minutes = (x // 100) % 100
    sec = x % 100
    return deg + minutes / 60 + sec / 3600


def season_of(dates):
    """Winter season = year of the spring in which the winter ends.
    Jul-Dec dates belong to the next year's season, e.g. freeze-up
    2001-11-20 and break-up 2002-05-10 are both season 2002."""
    return np.where(dates.dt.month >= 7, dates.dt.year + 1, dates.dt.year)


def days_between(later, earlier):
    return (later - earlier).dt.days


# ---- 1. Download ------------------------------------------------------------

print("Downloading ice observations ...")
ice = get_all_odata(f"{HYDRO_API}/JaatJaanlahto")

print("Downloading site information ...")
places = get_all_odata(f"{HYDRO_API}/Paikka")

# ---- 2. Clean ice events ----------------------------------------------------

ice_events = ice[ice["Lippu_id"].isin(EVENT_CODES.keys())].copy()
ice_events["date"] = pd.to_datetime(ice_events["Arvo"].str[:10], errors="coerce")
ice_events = ice_events.dropna(subset=["date"])
ice_events["event"] = ice_events["Lippu_id"].map(EVENT_CODES)
ice_events["season"] = season_of(ice_events["date"])
ice_events = ice_events.drop_duplicates(["Paikka_Id", "event", "date"])

if MIN_SEASON is not None:
    ice_events = ice_events[ice_events["season"] >= MIN_SEASON]

# A lake can freeze and thaw more than once in a winter. Keep:
#   break-up  -> the LAST full break-up of the season (final ice-off)
#   freeze-up -> the FIRST permanent freeze-up of the season
# and count the events, to flag unstable winters.
grouped = ice_events.groupby(["Paikka_Id", "season", "event"])["date"]
season_dates = grouped.agg(["min", "max", "count"]).reset_index()
is_breakup = season_dates["event"].str.startswith("breakup")
season_dates["chosen"] = np.where(is_breakup, season_dates["max"], season_dates["min"])
season_dates["chosen"] = pd.to_datetime(season_dates["chosen"])

dates_wide = season_dates.pivot_table(
    index=["Paikka_Id", "season"], columns="event", values="chosen", aggfunc="first"
)
counts_wide = season_dates.pivot_table(
    index=["Paikka_Id", "season"], columns="event", values="count", aggfunc="first"
)

season_data = pd.DataFrame(index=dates_wide.index)
for ev in EVENT_CODES.values():
    season_data[f"{ev}_date"] = (
        pd.to_datetime(dates_wide[ev]) if ev in dates_wide else pd.NaT
    )
    season_data[f"{ev}_n"] = counts_wide[ev] if ev in counts_wide else np.nan
season_data = season_data.reset_index()

# ---- 3. Derived outcome variables -------------------------------------------

jan1 = pd.to_datetime(season_data["season"].astype(str) + "-01-01")
sep1_prev = pd.to_datetime((season_data["season"] - 1).astype(str) + "-09-01")

# Break-up day of year (1 Jan = 1). Main outcome.
season_data["breakup_doy"] = days_between(season_data["breakup_date"], jan1) + 1
season_data["breakup_partial_doy"] = (
    days_between(season_data["breakup_partial_date"], jan1) + 1
)
# Freeze-up counted from 1 September of the autumn before (1 Sep = 1),
# so December and January freeze-ups stay on one continuous scale
season_data["freeze_day_from_sep1"] = (
    days_between(season_data["freeze_date"], sep1_prev) + 1
)
season_data["freeze_first_day_from_sep1"] = (
    days_between(season_data["freeze_first_date"], sep1_prev) + 1
)
# Ice-cover duration: permanent freeze-up to final break-up (days)
dur = days_between(season_data["breakup_date"], season_data["freeze_date"])
season_data["ice_duration"] = dur.where(dur > 0)

# Flags for checking the data before modelling
season_data["multiple_breakups"] = season_data["breakup_n"].fillna(0) > 1
season_data["multiple_freezes"] = season_data["freeze_n"].fillna(0) > 1
# Break-up outside roughly 1 March - 30 June is suspicious in Finland
season_data["breakup_suspicious"] = season_data["breakup_doy"].notna() & (
    (season_data["breakup_doy"] < 60) | (season_data["breakup_doy"] > 181)
)
season_data = season_data.drop(columns=[c for c in season_data if c.endswith("_n")])

# ---- 4. Select sites with long break-up records ------------------------------

with_breakup = season_data[season_data["breakup_doy"].notna()]
good_places = (
    with_breakup.groupby("Paikka_Id")
    .agg(
        n_seasons=("season", "nunique"),
        first_season=("season", "min"),
        last_season=("season", "max"),
        mean_breakup_doy=("breakup_doy", "mean"),
    )
    .reset_index()
)
freeze_counts = (
    season_data.groupby("Paikka_Id")["freeze_day_from_sep1"]
    .count()
    .rename("n_freeze_seasons")
    .reset_index()
)
good_places = good_places.merge(freeze_counts, on="Paikka_Id", how="left")
good_places = good_places[good_places["n_seasons"] >= MIN_SEASONS]

# ---- 5. Site information + coordinates --------------------------------------

site_info = places[places["Paikka_Id"].isin(good_places["Paikka_Id"])].copy()
site_info["Jarvi_Id"] = pd.to_numeric(site_info["Jarvi_Id"], errors="coerce").astype("Int64")
site_info["lat"] = dms_to_decimal(site_info["KoordLat"])
site_info["lon"] = dms_to_decimal(site_info["KoordLong"])
site_info = site_info.rename(
    columns={"KoordErTmPohj": "etrs_north", "KoordErTmIta": "etrs_east"}
)[
    ["Paikka_Id", "Nimi", "JarviNimi", "Jarvi_Id", "KuntaNimi", "PaaVesalNimi",
     "lat", "lon", "etrs_north", "etrs_east"]
]

# ---- 6. Lake characteristics from the Lakes API -----------------------------
# Fetched only for the lakes we need, in small batches.

lake_ids = (
    pd.to_numeric(site_info["Jarvi_Id"], errors="coerce").dropna().astype(int).unique()
)
lake_frames = []
print(f"Downloading lake information for {len(lake_ids)} lakes ...")
for i in range(0, len(lake_ids), 20):
    batch = lake_ids[i:i + 20]
    params = {
        "$filter": " or ".join(f"Jarvi_Id eq {j}" for j in batch),
        "$select": "Jarvi_Id,Vesiala,SyvyysKeski,SyvyysSuurin,Tilavuus,"
                   "KorkeusN2000,Rantaviiva",
    }
    try:
        lake_frames.append(get_all_odata(f"{LAKE_API}/Jarvi", params=params))
    except requests.RequestException as e:
        print(f"  Lake batch failed ({e}); continuing without it")

if lake_frames and any(len(f) for f in lake_frames):
    lakes = pd.concat(lake_frames, ignore_index=True)
    lake_info = pd.DataFrame({
        "Jarvi_Id": pd.to_numeric(lakes["Jarvi_Id"], errors="coerce").astype("Int64"),
        "lake_area_km2": pd.to_numeric(lakes["Vesiala"], errors="coerce") / 100,
        "lake_mean_depth_m": pd.to_numeric(lakes["SyvyysKeski"], errors="coerce"),
        "lake_max_depth_m": pd.to_numeric(lakes["SyvyysSuurin"], errors="coerce"),
        # thousand m3 -> million m3
        "lake_volume_mm3": pd.to_numeric(lakes["Tilavuus"], errors="coerce") / 1000,
        "lake_altitude_m": pd.to_numeric(
            lakes["KorkeusN2000"].astype(str).str.replace(",", "."), errors="coerce"
        ),
        "shoreline_km": pd.to_numeric(lakes["Rantaviiva"], errors="coerce"),
    })
    lake_info["log_lake_area"] = np.log(lake_info["lake_area_km2"].where(
        lake_info["lake_area_km2"] > 0))
    site_info = site_info.merge(lake_info, on="Jarvi_Id", how="left")
else:
    for col in ["lake_area_km2", "log_lake_area", "lake_mean_depth_m",
                "lake_max_depth_m", "lake_volume_mm3", "lake_altitude_m",
                "shoreline_km"]:
        site_info[col] = np.nan

good_places = good_places.merge(site_info, on="Paikka_Id", how="left").sort_values(
    "n_seasons", ascending=False
)

# ---- 7. Analysis dataset ----------------------------------------------------

analysis_data = season_data[season_data["Paikka_Id"].isin(good_places["Paikka_Id"])]
analysis_data = analysis_data.merge(site_info, on="Paikka_Id", how="left")

# Time centred at 1990 and measured in decades, so the trend coefficient is
# "days per decade" and the intercept refers to 1990
analysis_data["decade_c"] = (analysis_data["season"] - 1990) / 10
analysis_data["lat_c"] = analysis_data["lat"] - analysis_data["lat"].mean()

columns = [
    "Paikka_Id", "Nimi", "JarviNimi", "Jarvi_Id", "KuntaNimi", "PaaVesalNimi",
    "season", "decade_c",
    "breakup_date", "breakup_doy",
    "breakup_partial_date", "breakup_partial_doy",
    "freeze_date", "freeze_day_from_sep1",
    "freeze_first_date", "freeze_first_day_from_sep1",
    "ice_duration",
    "lat", "lon", "lat_c", "etrs_north", "etrs_east",
    "lake_area_km2", "log_lake_area", "lake_mean_depth_m", "lake_max_depth_m",
    "lake_volume_mm3", "lake_altitude_m", "shoreline_km",
    "multiple_breakups", "multiple_freezes", "breakup_suspicious",
]
analysis_data = analysis_data[columns].sort_values(["Paikka_Id", "season"])

# Dates as plain YYYY-MM-DD in the CSV
for col in [c for c in analysis_data if c.endswith("_date")]:
    analysis_data[col] = analysis_data[col].dt.strftime("%Y-%m-%d")

# ---- 8. Quick checks --------------------------------------------------------

print(f"Sites kept:           {len(good_places)}")
print(f"Site-seasons:         {len(analysis_data)}")
print(f"With break-up date:   {analysis_data['breakup_doy'].notna().sum()}")
print(f"With freeze-up date:  {analysis_data['freeze_day_from_sep1'].notna().sum()}")
print(f"With ice duration:    {analysis_data['ice_duration'].notna().sum()}")
print(f"Suspicious break-ups: {analysis_data['breakup_suspicious'].sum()}")
print(f"Sites missing lake info: {good_places['lake_area_km2'].isna().sum()}")
print(analysis_data[["breakup_doy", "freeze_day_from_sep1",
                     "ice_duration", "lat"]].describe().round(1))

# ---- 9. Save ----------------------------------------------------------------
# utf-8-sig keeps ä/ö correct when the file is opened in Excel

analysis_data.to_csv("/Users/marino/Downloads/icebreak_data.csv", index=False, encoding="utf-8-sig")
good_places.to_csv("/Users/marino/Downloads/icebreak_data_places.csv", index=False, encoding="utf-8-sig")
print("Saved icebreak_data.csv and icebreak_data_places.csv")

Sites kept:           107
Site-seasons:         9572
With break-up date:   9244
With freeze-up date:  7928
With ice duration:    7664
Suspicious break-ups: 0
Sites missing lake info: 3
       breakup_doy  freeze_day_from_sep1  ice_duration     lat
count       9244.0                7928.0        7664.0  9572.0
mean         130.5                  84.8         168.1    63.0
std           12.5                  20.4          28.8     2.1
min           64.0                  32.0          16.0    60.2
25%          123.0                  71.0         151.0    61.5
50%          130.0                  84.0         168.0    62.5
75%          137.0                  97.0         186.0    63.6
max          181.0                 189.0         264.0    69.8
Saved icebreak_data.csv and icebreak_data_places.csv
